# Modelagem Multidomínio DBEM 2D com Leis Coesivas de Interface

Este Jupyter Notebook apresenta o fluxo de trabalho (**pipeline**) completo para a simulação de fratura e descolamento de interface (*debonding*) em juntas coladas e estruturas compósitas utilizando o **Método dos Elementos de Contorno Dual com Zonas Coesivas (Multizone Cohesive DBEM 2D)**.

O objetivo deste material didático é guiar estudantes e pesquisadores passo a passo no processo de:
1. **Configuração da Geometria e Malha**: criação do arquivo `.geo` paramétrico e geração da malha `.msh` com o **Gmsh**.
2. **Pré-processamento**: leitura da malha via **`meshio`** e exportação automática dos 5 arquivos de entrada formatados para o solver Fortran.
3. **Inspeção Visual**: conferência da malha, sub-regiões e condições de contorno (Dirichlet e Neumann).
4. **Compilação**: compilação do código Fortran utilizando `gfortran`, `mingw32-make` e bibliotecas `LAPACK`/`BLAS`.
5. **Execução Numérica**: resolução do problema não linear incremental via Newton-Raphson com controle de deslocamento.
6. **Pós-processamento e Gráficos**: visualização das curvas de resposta estrutural (Força $\times$ Deslocamento) e das leis de tração-separação coesivas.

---
> **Artigo de Referência**: *Comparative analysis of adhesive debonding in three-point bending tests using boundary element methods: Coupled criterion versus interface damage model*, Engineering Analysis with Boundary Elements (EABE), 2025.

## 0. Preparação do Ambiente Virtual (.venv) e Instalação de Dependências

Antes de iniciar a simulação, esta célula:
1. **Cria o ambiente virtual (`.venv`) local**, caso ainda não exista, e instala as dependências (`numpy`, `matplotlib`, `meshio`, `ipykernel`).
2. **Verifica e instala os pacotes no kernel atual**, garantindo que as próximas células executem de imediato sem erro de `ModuleNotFoundError` mesmo se o kernel ativo no VSCode for outro interpretador Python.


In [ ]:
import os
import sys
import subprocess
import importlib

# 1. Lista de pacotes necessários para a simulação e pós-processamento
required_packages = ['numpy', 'matplotlib', 'meshio']

# 2. Localização ou criação do ambiente virtual (.venv)
possible_venvs = [os.path.abspath('.venv'), os.path.abspath('../.venv')]
venv_dir = next((v for v in possible_venvs if os.path.exists(v)), os.path.abspath('.venv'))
venv_scripts = os.path.join(venv_dir, 'Scripts') if os.name == 'nt' else os.path.join(venv_dir, 'bin')
venv_python = os.path.join(venv_scripts, 'python.exe' if os.name == 'nt' else 'python')

if not os.path.exists(venv_dir):
    print(f'[*] Criando ambiente virtual (.venv) em: {venv_dir}...')
    try:
        subprocess.check_call([sys.executable, '-m', 'venv', venv_dir])
        print('[+] Ambiente virtual (.venv) criado com sucesso!')
        if os.path.exists(venv_python):
            print('[*] Instalando dependências e ipykernel dentro do .venv...')
            subprocess.check_call([venv_python, '-m', 'pip', 'install', 'ipykernel', *required_packages])
            print('[+] Ambiente virtual (.venv) totalmente configurado!')
    except Exception as e:
        print(f'[!] Aviso ao criar .venv: {e}')
else:
    print(f'[+] Ambiente virtual (.venv) detectado em: {venv_dir}')
    if os.path.exists(venv_python):
        try:
            subprocess.run([venv_python, '-m', 'pip', 'install', 'ipykernel', *required_packages],
                           capture_output=True, check=True)
            print('[+] Dependências do .venv verificadas com sucesso!')
        except Exception:
            pass

# 3. Garante que os pacotes necessários estejam instalados no kernel ATUALMENTE em execução
# (Isso garante que a próxima célula execute imediatamente sem erro, mesmo se o usuário ainda não tiver trocado o kernel no VSCode)
missing_packages = []
for pkg in required_packages:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing_packages.append(pkg)

if missing_packages:
    print(f'[*] Instalando pacotes ausentes no kernel atual ({sys.executable}): {missing_packages}...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
        importlib.invalidate_caches()
        print('[+] Pacotes instalados com sucesso no kernel atual!')
    except Exception as e:
        print(f'[!] Tentativa com sys.executable falhou: {e}')
        try:
            from IPython import get_ipython
            ipy = get_ipython()
            if ipy is not None:
                print('[*] Tentando instalação via magic %pip do IPython...')
                ipy.run_line_magic('pip', f'install {" ".join(missing_packages)}')
                importlib.invalidate_caches()
                print('[+] Pacotes instalados via %pip!')
        except Exception as e_magic:
            print(f'[!] Erro no fallback %pip: {e_magic}')
else:
    print('[+] Todos os pacotes necessários já estão disponíveis no ambiente atual.')

print('\n💡 Dica no VSCode: Caso deseje utilizar o ambiente virtual isolado, clique no seletor de kernel no canto')
print('   superior direito do VSCode (\'Select Kernel\' -> \'Python Environments\') e escolha o Python do .venv.')


## 1. Configuração do Ambiente e Importações

Configuramos o ambiente local para garantir que o compilador Fortran (`gfortran`), o utilitário `mingw32-make` e o `gmsh` estejam acessíveis no `PATH` do sistema.

In [ ]:
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
from IPython.display import Image, display

# Adiciona compilador MinGW-w64 (MSYS2 UCRT64) e scripts locais ao PATH do processo
ucrt64_bin = r"C:\msys64\ucrt64\bin"
python_scripts = os.path.join(os.path.dirname(sys.executable), "Scripts")

for p in [ucrt64_bin, python_scripts]:
    if os.path.exists(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

print("Ambiente configurado com sucesso!")
print(f"Diretório de execução: {os.getcwd()}")

## 2. Definição da Geometria no Gmsh (`.geo`)

A geometria do problema é definida através de um arquivo `.geo` do Gmsh. O modelo considerado consiste em uma placa de dimensões $0.1 \times 0.1$ m dividida diagonalmente em duas sub-regiões triangulares unidas por uma **interface coesiva**:
- **Sub-região 1 ($\Omega_1$)**: Triângulo inferior, delimitado pelos nós 1 a 6.
- **Sub-região 2 ($\Omega_2$)**: Triângulo superior, delimitado pelos nós 7 a 9.
- **Interfaces Coesivas**:
  - `interface_m` ($I^-$): lado da interface pertencente à Sub-região 1 (do nó 4 ao nó 5).
  - `interface_p` ($I^+$): lado da interface pertencente à Sub-região 2 (do nó 9 ao nó 7).

> **Dica para o Aluno**: Para analisar uma geometria diferente (como juntas de sobreposição simples, vigas coladas ou corpos com trincas), basta editar as coordenadas e linhas no bloco abaixo.

In [ ]:
geo_file_name = "modelo_geometria"

geo_code = """// ==============================================================================
// Definição Paramétrica Gmsh - DBEM 2D Coesivo Multidomínio
// ==============================================================================
ref = 0.1;

// Nós da Sub-região 1 (Triângulo Inferior)
Point(1) = {0.000, 0.000, 0, ref};
Point(2) = {0.005, 0.000, 0, ref};
Point(3) = {0.095, 0.000, 0, ref};
Point(4) = {0.100, 0.000, 0, ref};
Point(5) = {0.000, 0.100, 0, ref};
Point(6) = {0.000, 0.095, 0, ref};

// Linhas da Sub-região 1 (Sentido anti-horário)
Line(1) = {1, 2};
Line(2) = {2, 3};
Line(3) = {3, 4};
Line(4) = {4, 5}; // Interface Im (lado -)
Line(5) = {5, 6};
Line(6) = {6, 1};

Curve Loop(1) = {1, 2, 3, 4, 5, 6};
Plane Surface(1) = {1};

// Nós da Sub-região 2 (Triângulo Superior)
Point(7) = {0.100, 0.000, 0, ref};
Point(8) = {0.100, 0.100, 0, ref};
Point(9) = {0.000, 0.100, 0, ref};

// Linhas da Sub-região 2 (Sentido anti-horário)
Line(7) = {7, 8};
Line(8) = {8, 9};
Line(9) = {9, 7}; // Interface Ip (lado +)

Curve Loop(2) = {7, 8, 9};
Plane Surface(2) = {2};

// Discretização linear (1 elemento por segmento)
Transfinite Curve {1, 2, 3, 4, 5, 6, 7, 8, 9} = 2 Using Progression 1;

// Grupos Físicos de Contorno (1D)
Physical Curve("bot") = {1, 2, 3};
Physical Curve("interface_m") = {4};
Physical Curve("left") = {5, 6};
Physical Curve("right") = {7};
Physical Curve("top") = {8};
Physical Curve("interface_p") = {9};

// Grupos Físicos de Superfície (2D)
Physical Surface("subregion_1") = {1};
Physical Surface("subregion_2") = {2};
"""

with open(f"{geo_file_name}.geo", "w", encoding="utf-8") as f:
    f.write(geo_code)

print(f"Arquivo de geometria '{geo_file_name}.geo' gravado com sucesso!")

## 3. Geração da Malha com Gmsh (`.msh`)

Executamos o Gmsh para gerar os elementos lineares de contorno e a malha 2D associada.

In [ ]:
from gmsh_to_dbem import generate_geometry_and_mesh

# Executa o Gmsh gerando o arquivo .msh
generate_geometry_and_mesh(geo_file_name, geo_code)
print(f"Arquivo de malha gerado: '{geo_file_name}.msh'")

## 4. Definição dos Materiais, Condições de Contorno e Parâmetros de Análise

Definimos os parâmetros físicos e numéricos através de dicionários Python intuitivos:
- **Materiais**: $E$ e $\nu$ por sub-região, e parâmetros coesivos de Modo I ($K_{n0}, \Phi_n, \sigma_{max}$) e Modo II ($K_{t0}, \Phi_t, \tau_{max}$).
- **Condições de Contorno**: prescrições associadas a cada `Physical Curve`.
- **Parâmetros Numéricos**: passos de carga ($N_s$), tolerância de Newton-Raphson e grau de liberdade controlado.

In [ ]:
# 1. Materiais (Sub-regiões e Interface Coesiva)
materials = {
    "subregions": {
        1: {"E": 32.0e3, "nu": 0.2},   # Sub-região 1 (MPa)
        2: {"E": 32.0e3, "nu": 0.2},   # Sub-região 2 (MPa)
    },
    "cohesive_mode_I": {
        "Kn0": 1.0e9,       # Rigidez inicial normal (MPa/m)
        "phin": 0.0001,     # Energia de fratura Modo I (MPa·m)
        "smax": 4.0,        # Tensão normal de pico (MPa)
    },
    "cohesive_mode_II": {
        "Kto": 1.0e9,       # Rigidez inicial tangencial (MPa/m)
        "phit": 0.0002,     # Energia de fratura Modo II (MPa·m)
        "tmax": 3.0,        # Tensão cisalhante de pico (MPa)
    }
}

# 2. Condições de Contorno por Physical Curve
# Formato: (A, B, C) onde o valor prescrito é A*S^2 + B*S + C ao longo do elemento
bound_cond = {
    "bot": {
        "surface_type": "B",
        "Ux": None,
        "Uy": (0.0, 0.0, 0.0),     # Borda inferior: apoio vertical Uy = 0
    },
    "left": {
        "surface_type": "B",
        "Ux": (0.0, 0.0, 0.0),     # Borda esquerda: engastamento horizontal Ux = 0
        "Uy": None,
    },
    "right": {
        "surface_type": "B",
        "Tx": (0.0, 0.0, -1.0),    # Borda direita: tração Tx = -1.0 MPa
        "Ty": None,
    },
    "top": {
        "surface_type": "B",
        "Tx": None,
        "Ty": (0.0, 0.0, 1.0),     # Borda superior: tração Ty = +1.0 MPa
    },
    "interface_m": {
        "surface_type": "Im",      # Interface lado menos
        "Ux": None, "Uy": None, "Tx": None, "Ty": None,
    },
    "interface_p": {
        "surface_type": "Ip",      # Interface lado mais
        "Ux": None, "Uy": None, "Tx": None, "Ty": None,
    },
}

# Prescrição pontual complementar (canto inferior esquerdo)
element_specific_bc = {
    1: {"Ux": (0.0, 0.0, 0.0)}
}

# 3. Parâmetros Numéricos de Análise
analysis_params = {
    "N_GAUSS": 400,
    "CP_DISTANCING": 0.0,
    "PLANE_STATE": "PT",      # PT = Deformação Plana
    "ANALYSIS_TYPE": "CP",    # CP = Propagação Coesiva
    "Ns": 50,                 # 50 passos incrementais
    "tol_Newton": 1.0e-2,     # Tolerância Newton-Raphson
    "itemax": 100000,         # Número máximo de iterações
    "structural_node": 9,     # Nó monitorado da resposta estrutural
    "cohesive_node": 18,      # Nó monitorado da interface coesiva
    "control": {
        "type": "U",
        "node": 17,           # Nó com controle de deslocamento
        "direction": 2,       # Direção Y
        "value": 0.00012,     # Deslocamento total / incremento
    }
}

print("Configurações do modelo definidas com sucesso.")

## 5. Conversão da Malha e Geração dos Arquivos de Entrada Fortran

Utilizamos o módulo `meshio` para extrair os elementos de contorno, identificar as sub-regiões e formatar os 5 arquivos de entrada exigidos pelo solver Fortran:
- `1_BOUNDARY_MESH.txt`
- `2_MATERIALS.txt`
- `3_BOUNDARY_CONDITIONS.txt`
- `4_ANALYSIS_PARAMETERS.txt`
- `5_POSTPROCESSING_MESH.txt`

In [ ]:
from gmsh_to_dbem import compute_inodes, create_fortran_input_files

inp_config = {
    "materials": materials,
    "bound_cond": bound_cond,
    "element_specific_bc": element_specific_bc,
    "analysis_params": analysis_params
}

# Leitura topológica via meshio
comp_data = compute_inodes(geo_file_name, bound_cond)
print(f"Nós de contorno extraídos: {len(comp_data['coordinates'])}")
print(f"Elementos lineares 1D: {len(comp_data['line_elements'])}")
print(f"Sub-regiões identificadas: {comp_data['n_subregions']}")

# Escrita dos arquivos de entrada no diretório local
create_fortran_input_files(".", comp_data, inp_config)
print("\n[OK] Todos os arquivos 1_*.txt a 5_*.txt foram gerados com sucesso!")

## 6. Visualização da Malha e Condições de Contorno

Inspecionamos visualmente a malha de elementos de contorno, as sub-regiões, o destaque da interface coesiva e os marcadores de condições de contorno (Dirichlet em vermelho, Neumann em azul e nó de controle em verde).

In [ ]:
from plot_mesh_bc import read_mesh_and_bcs, show_dbem_geometry_and_bcs

# Carrega dados gerados e desenha a figura do problema
data_plot = read_mesh_and_bcs(".")
fig_name = "mesh_and_boundary_conditions.png"
show_dbem_geometry_and_bcs(data_plot, fig_name)

# Exibe a imagem inline no notebook
display(Image(fig_name, width=650))

## 7. Compilação do Código Fortran (`MAIN_COHESIVE_DBEM_2D`)

Compilamos o código Fortran via `makefile` utilizando `mingw32-make` (no Windows) ou `make` (no Linux), com aceleração OpenMP e ligação com `LAPACK`/`BLAS`.

In [ ]:
# Identifica o comando do makefile disponível
ucrt_make = r"C:\msys64\ucrt64\bin\mingw32-make.exe"
make_cmd = ucrt_make if os.path.exists(ucrt_make) else ("mingw32-make" if os.name == "nt" else "make")

print(f"Compilando solver com '{make_cmd}'...")
res_make = subprocess.run(make_cmd, env=os.environ, capture_output=True, text=True)
print(res_make.stdout)

if res_make.returncode == 0:
    print("[OK] Executável compilado com sucesso!")
else:
    print("Aviso/Erro na compilação:")
    print(res_make.stderr)

## 8. Execução da Simulação Numérica

Executamos o solver `MAIN_COHESIVE_DBEM_2D`, que realiza os 50 passos de propagação coesiva através de iterações de Newton-Raphson com controle de deslocamento.

In [ ]:
exe_path = os.path.abspath("MAIN_COHESIVE_DBEM_2D.exe") if os.name == "nt" else "./MAIN_COHESIVE_DBEM_2D"
print(f"Executando solver: {exe_path}...")

res_solver = subprocess.run(exe_path, env=os.environ, capture_output=True, text=True)
print(res_solver.stdout)

if "SOLUTION IS DONE" in res_solver.stdout:
    print("[OK] Simulação numérica finalizada com sucesso!")
else:
    print("Aviso durante a execução:")
    print(res_solver.stderr)

## 9. Pós-processamento e Curvas de Resposta

Lemos os arquivos de saída gerados pelo solver e plotamos:
1. **Resposta Estrutural Global ($T_y \times U_y$)**: curva de reação versus deslocamento, mostrando o regime elástico, o pico de carga e o amolecimento (*softening*) por propagação de fratura.
2. **Lei Coesiva Interfacial ($\tau_t \times \Delta u_t$)**: relação tração-separação na interface de colagem.
3. **Degradação da Rigidez Interfacial**: evolução das rigidezes tangencial ($K_{tt}$) e normal ($K_{nn}$) ao longo dos passos de carga.

In [ ]:
# Leitura dos arquivos de saída numéricos
data_struct = np.loadtxt("8_OUTPUT_NODAL_STRUCTURAL_RESPONSE.txt", skiprows=1)
data_coh = np.loadtxt("9_OUTPUT_COHESIVE_RESPONSE.txt", skiprows=1)

Ux, Uy, Tx, Ty = data_struct[:, 0], data_struct[:, 1], data_struct[:, 2], data_struct[:, 3]
Knn, Knt, Ktn, Ktt, Dun, Dut = [data_coh[:, i] for i in range(6)]
tau_t = Ktt * Dut
sigma_n = Knn * Dun
steps = np.arange(1, len(Ty) + 1)

# Construção do painel gráfico comparativo
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.8), dpi=150)

# Gráfico 1: Resposta Estrutural Global
ax1.plot(Uy * 1e3, Ty, "b-o", markersize=3.5, lw=1.8, label="Reação $T_y$")
ax1.set_xlabel("Deslocamento $U_y$ ($10^{-3}$ m)", fontweight="bold")
ax1.set_ylabel("Tração de Reação $T_y$ (MPa)", fontweight="bold")
ax1.set_title("Resposta Estrutural Global", fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.45)
ax1.legend()

# Gráfico 2: Lei Coesiva Interfacial
ax2.plot(Dut * 1e3, tau_t, "r-s", markersize=3.5, lw=1.8, label=r"Tração Coesiva $\tau_t$")
ax2.set_xlabel(r"Salto Tangencial $\Delta u_t$ ($10^{-3}$ m)", fontweight="bold")
ax2.set_ylabel(r"Tração Coesiva $\tau_t$ (MPa)", fontweight="bold")
ax2.set_title("Lei de Tração-Separação", fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.45)
ax2.legend()

# Gráfico 3: Degradação da Rigidez
ax3.semilogy(steps, Ktt, "g-^", markersize=3.5, lw=1.8, label="Rigidez $K_{tt}$")
ax3.semilogy(steps, Knn, "m-v", markersize=3.5, lw=1.8, label="Rigidez $K_{nn}$")
ax3.set_xlabel("Passo de Carga (Step)", fontweight="bold")
ax3.set_ylabel("Rigidez Tangente (MPa/m)", fontweight="bold")
ax3.set_title("Degradação da Rigidez Interfacial", fontweight="bold")
ax3.grid(True, linestyle="--", alpha=0.45)
ax3.legend()

plt.tight_layout()
results_fig = "cohesive_results.png"
plt.savefig(results_fig, dpi=300, bbox_inches="tight")
plt.close(fig)

# Exibe a imagem de resultados gerada
display(Image(results_fig, width=900))
print(f"Painel gráfico salvo com sucesso em '{results_fig}'.")

## 10. Conclusão e Guia Rápido para Novos Modelos

O pipeline completo foi executado do início ao fim com sucesso!

Para analisar um **novo problema ou geometria diferente**:
1. Edite o bloco da **Etapa 2** definindo os pontos, linhas e superfícies no `.geo`.
2. Ajuste as propriedades na **Etapa 4** (`materials`, `bound_cond` e `analysis_params`).
3. Execute o notebook sequencialmente (`Kernel -> Restart & Run All`).